## **RunnableWithMessageHistory**

**RunnableWithMessageHistory** is a higher-order Runnable wrapper that intercepts config to load/add chat history before chain execution and saves the new messages after.

**Syntax: Defining RunnableWithMessageHistory**
```python
chat_with_history = RunnableWithMessageHistory(
    runnable: Runnable,                                            # Your chain
    get_session_history: Callable[[str], BaseChatMessageHistory],  # Pass the chat history store
    input_messages_key: str = "input",               # Key in input to extract new message(s)
    history_messages_key: str = "chat_history",      # Key to inject history in input
    history_factory_config: list = [                 # Defines keys to be passed into "get_message_history"
        ConfigurableFieldSpec(
            id="session_id",                         # The unique identifier of the field
            annotation=str,                          # Annotation/type for the field
            name="Session ID",
            description="Unique identifier for a session.",
        ),
    ]
)
```
- `RunnableWithMessageHistory` wraps another `Runnable` and manages the chat message history for it; it is responsible for reading and updating the chat message history.
- `ConfigurableFieldSpec` allows you to define the unique ID like `session_id` (by default). You can update the ID to `thread_id` using the syntax above.

**Syntax: Calling RunnableWithMessageHistory**
```python
chat_with_history.invoke(..., config={"configurable": {"session_id": "bar"}}))
```
- `RunnableWithMessageHistory` must always be **called with a config** that contains the appropriate parameters for the chat message history factory.
- **Important:** If "history_factory_config" is not passed, by default, the `Runnable` is **expected to take a single configuration parameter called `session_id` which is a string**. This parameter is used to create a new or look up an existing chat message history that matches the given `session_id`.


### **Step 1: Setup prompt with history slot**

In [1]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful. Use chat history."),
    MessagesPlaceholder("chat_history"),  # Dynamic insertion point
    ("human", "{input}")
])

### **Step 2: Initialize a Chat Model and Output Parser**

In [2]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

chat_model = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-4o-mini",
    temperature=0.0
)

In [3]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

### **Step 3: Define a Base Chain**

In [4]:
chain = prompt | chat_model | output_parser

### **Step 4: Define a Persistent Layer**

In [5]:
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory

# _store with the schema: 
# str - session_id
# BaseChatMessageHistory - for storing chat message history
_store: dict[str, BaseChatMessageHistory] = {}

def get_history(
    session_id: str
) -> BaseChatMessageHistory:
    if session_id not in _store:
        _store[session_id] = InMemoryChatMessageHistory()
    return _store[session_id]

### **Step 5: Add Memory to the Base Chain using RunnableWithMessageHistory**

In [10]:
from langchain_core.runnables.history import RunnableWithMessageHistory, ConfigurableFieldSpec

chain_with_history = RunnableWithMessageHistory(
    runnable=chain,
    get_session_history=get_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="thread_id",
            annotation=str
        ),
    ]
)

### **Step 6: Configure session_id and Invoke the Base Chain with Memory repeatedly**

In [11]:
config_1 = {"configurable": {"thread_id": "abc123"}}

chain_with_history.invoke(
    {"input": "Hi, Kanav this side."}, config_1
)

'Hi Kanav! How can I assist you today?'

In [12]:
config_2 = {"configurable": {"session_id": "abc456"}}

chain_with_history.invoke({"input": "Hi, Bansal this side."}, config_2)

ValueError: Missing keys ['thread_id'] in config['configurable'] Expected keys are ['thread_id'].When using via .invoke() or .stream(), pass in a config; e.g., chain.invoke({'input': 'foo'}, {'configurable': {'thread_id': '[your-value-here]'}})

In [9]:
chain_with_history.invoke({"input": "What is my name?"}, config_1)

'Your name is Kanav. How can I help you today?'

In [10]:
chain_with_history.invoke({"input": "What is my name?"}, config_2)

'Your name is Bansal. How can I help you today?'

### **Let's analyse the Memory Layer**

In [18]:
chat_history = get_history(session_id="abc123")

for msg in chat_history.messages:
    msg.pretty_print()

================================ Human Message =================================

Hi, Kanav this side.
================================== Ai Message ==================================

Hi Kanav! How can I assist you today?
================================ Human Message =================================

What is my name?
================================== Ai Message ==================================

Your name is Kanav. How can I help you today?


In [19]:
chat_history = get_history(session_id="abc456")

for msg in chat_history.messages:
    msg.pretty_print()

================================ Human Message =================================

Hi, Bansal this side.
================================== Ai Message ==================================

Hello, Bansal! How can I assist you today?
================================ Human Message =================================

What is my name?
================================== Ai Message ==================================

Your name is Bansal. How can I help you today?


### **Example where the session factory takes two keys (`user_id` and `conversation_id`)**

In [ ]:
store = {}


def get_history(
    user_id: str, conversation_id: str
) -> BaseChatMessageHistory:
    if (user_id, conversation_id) not in store:
        store[(user_id, conversation_id)] = InMemoryChatMessageHistory()
    return store[(user_id, conversation_id)]


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You're an assistant who's good at {ability}"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)

chain = prompt | ChatAnthropic(model="claude-2")


with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history=get_session_history,
    input_messages_key="question",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="user_id",
            annotation=str,
            name="User ID",
            description="Unique identifier for the user.",
            default="",
            is_shared=True,
        ),
        ConfigurableFieldSpec(
            id="conversation_id",
            annotation=str,
            name="Conversation ID",
            description="Unique identifier for the conversation.",
            default="",
            is_shared=True,
        ),
    ],
)

with_message_history.invoke(
    {"ability": "math", "question": "What does cosine mean?"},
    config={"configurable": {"user_id": "123", "conversation_id": "1"}},
)